<a href="https://colab.research.google.com/github/OlaYounisACCA/SDS/blob/Dev/Lab4_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Strategic Data Science Lab 4 - Predictive Modelling for Finance

### 🧠**Overview**
Your role: Senior Finance Data Scientist, NovaCore Solutions Ltd.

The Finance Director wants better evidence for revenue planning by answering two questions:

Which business factors help explain and predict monthly revenue?
How accurately can future revenue be forecast?
The dataset is already clean (preparation was covered in Lab 2).

The dataset has already been prepared for modelling. You are **not expected to repeat the data-cleaning activities**.

## 1. Lab 4 Learning Journey

**Financial history → Decomposition → two modelling questions**

- **Revenue prediction:** Linear Regression using business drivers.
- **Revenue forecasting:** Seasonal Naïve, Holt-Winters and Prophet.

Both branches meet at **validation and model comparison**, leading to the final **Finance Decision**.

# 2. Setup & Environment Check

We use:

- **pandas** for tabular financial data;
- **numpy** for numerical calculations;
- **matplotlib** for visualisation;
- **scikit-learn** for linear regression and evaluation;
- **statsmodels** for decomposition and Holt-Winters;
- **Prophet** for alternative forecasting.

The code comments explain the analytical purpose of important steps.

In [ ]:
# Import pandas for loading and analysing financial data.
import pandas as pd

# Import numpy for numerical calculations.
import numpy as np

# Import matplotlib for charts.
import matplotlib.pyplot as plt

# Import linear regression and evaluation metrics.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Import time-series decomposition.
from statsmodels.tsa.seasonal import seasonal_decompose

# Import Holt-Winters exponential smoothing.
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Import FuncFormatter to ensure proper axis format on plots
from matplotlib.ticker import FuncFormatter

print("pandas version:", pd.__version__)

## 3. Install Prophet

In Google Colab or another compatible Python environment, run the cell below to install and import Prophet.

If you are using Try JupyterLab (JupyterLite), skip this cell: Prophet cannot be installed there. If installation fails in another environment, skip this cell too. Continue with Section 4; Section 12 will explain which Prophet activities to skip.

In [ ]:
# Install Prophet for a new Colab/runtime environment.
# If it is already installed, the command normally completes quickly.
%pip install prophet -q

# Import Prophet after installation.
from prophet import Prophet

print("Prophet is ready.")

# 4. Load the Clean Financial Dataset
The dataset contains one observation per month from January 2021 to December 2025.

**Important:** The dataset has already been prepared for modelling. Do not repeat the data-cleaning activities from Lab 2.

In [ ]:
# Load the clean NovaCore financial history dataset.
df = pd.read_csv("NovaCore_Lab4_Financial_History.csv")

# Convert the month field to datetime so it can be treated as a time series.
df["month"] = pd.to_datetime(df["month"])

# Sort chronologically and reset the row index.
df = df.sort_values("month").reset_index(drop=True)

# Display the first five observations.
df.head()

In [ ]:
# Confirm the time period and dataset dimensions.
print("First month:", df["month"].min().date())
print("Last month:", df["month"].max().date())
print("Rows:", len(df))
print("Columns:", len(df.columns))

In [ ]:
# Confirm that the modelling dataset is clean.
print("Missing cells:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate months:", df["month"].duplicated().sum())
print("Chronologically ordered:", df["month"].is_monotonic_increasing)

# Create a sequential month index for the regression trend feature.
if "month_index" not in df.columns:
    df["month_index"] = np.arange(len(df))

# 5. Explore NovaCore's Financial Performance

In [ ]:
# Summarise the main financial variables.
df[[
    "revenue_CUR",
    "operating_cost_CUR",
    "gross_profit_CUR",
    "net_cash_flow_CUR"
]].describe().T

In [ ]:

# Plot monthly revenue across the complete historical period.
# Create a 14 × 5 inch figure for the revenue chart.
plt.figure(figsize=(14, 5))

# Plot month on the x-axis and revenue on the y-axis.
# marker="o" shows each monthly observation; linewidth controls line thickness.
plt.plot(df["month"], df["revenue_CUR"], marker="o", linewidth=2)

# Add the chart title and axis labels.
plt.title("NovaCore Monthly Revenue — Full Historical Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

# Select every second month to keep the x-axis readable.
ticks = df["month"].iloc[::2]

# Format the dates as "Jan 21" and rotate them for readability.
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

# Adjust spacing automatically and display the chart.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))
plt.tight_layout()


# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )

In [ ]:
# Plot monthly net cash flow.
plt.figure(figsize=(12, 5))
plt.plot(df["month"], df["net_cash_flow_CUR"], marker="o")

# Add a zero reference line to distinguish positive and negative cash flow (if any).
plt.axhline(0, linewidth=1)

plt.title("NovaCore Monthly Net Cash Flow — Full Historical Period")
plt.xlabel("Month")
plt.ylabel("Net Cash Flow (CUR)")

ticks = df["month"].iloc[::2]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))
plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()

# 6. Decompose the Revenue Time Series

## Why decomposition comes first

Before building either regression or forecasting models, understand the structure of the revenue series.

Decomposition separates the observed series into:

- **Trend** — longer-term movement;
- **Seasonality** — recurring patterns;
- **Residuals** — variation not explained by the extracted components.

This provides a visual foundation for the modelling that follows.

In [ ]:
# Create a monthly time-indexed revenue series.
revenue_series = (
    df.set_index("month")["revenue_CUR"]
    .asfreq("ME")
)

# Decompose the revenue series using an additive model.
# A period of 12 represents annual seasonality in monthly data.
# extrapolate_trend="freq" estimates the trend at the beginning
# and end of the series where a centred moving average cannot
# be calculated directly.
decomposition = seasonal_decompose(
    revenue_series,
    model="additive",
    period=12,
    extrapolate_trend="freq"
)

print("Revenue decomposition completed.")

In [ ]:
from matplotlib.ticker import FuncFormatter

# Create four vertically stacked plots and share the same x-axis.
fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

# Plot the original revenue series.
decomposition.observed.plot(ax=axes[0])
axes[0].set_title("Observed Revenue")
axes[0].set_ylabel("Revenue")

# Plot the estimated long-term trend.
decomposition.trend.plot(ax=axes[1])
axes[1].set_title("Trend")
axes[1].set_ylabel("Trend")

# Plot the recurring seasonal pattern.
decomposition.seasonal.plot(ax=axes[2])
axes[2].set_title("Seasonality")
axes[2].set_ylabel("Seasonal")

# Plot the remaining unexplained variation.
decomposition.resid.plot(ax=axes[3])
axes[3].set_title("Residuals")
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Month")

# For all components axes, do the following:
for ax in axes:
  #Format all financial values in millions with two decimal places.
    ax.yaxis.set_major_formatter(
        FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
    )

    # Add minor gridlines.
    ax.minorticks_on()
    ax.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

    # Add organization name to the lower-right part of each subplot.
    ax.text(
        0.99, 0.01, "ACCA",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        fontstyle="italic",
        color="#E31837"
    )

# Use every second month to keep the shared x-axis readable.
ticks = decomposition.observed.index[::2]
axes[3].set_xticks(ticks)
axes[3].set_xticklabels(
    ticks.strftime("%b %y"),
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.show()

### Interpretation

Consider:

1. Is there an overall trend?
2. Is there a recurring seasonal pattern?
3. Are residuals centred around zero?
4. Are some periods more difficult to explain?

The decomposition gives us evidence about the structure that forecasting models may need to capture.

# 7. Explore Potential Revenue Drivers

Now move from the time structure of revenue to the business drivers that may help explain it.

Candidate drivers include:

- customer orders;
- active customers;
- customer retention;
- sales pipeline;
- marketing spend;
- capacity utilisation.

**Correlation indicates association; it does not prove causation.**

In [ ]:
# Select revenue and the business drivers to investigate.
revenue_variables = [
    "revenue_CUR",
    "customer_orders",
    "active_customers",
    "customer_retention_pct",
    "sales_pipeline_CUR",
    "marketing_spend_CUR",
    "capacity_utilisation_pct"
]

# Calculate pairwise correlations between the selected variables.
correlation_matrix = df[revenue_variables].corr()

correlation_matrix

In [ ]:
# Extract revenue correlations, exclude revenue itself, and sort them.
revenue_correlations = correlation_matrix["revenue_CUR"].drop("revenue_CUR").sort_values()

# Plot the correlations as a horizontal bar chart.
ax = revenue_correlations.plot(kind="barh", figsize=(9, 5))
ax.set_title("Correlation with Revenue")
ax.set_xlabel("Correlation")
ax.set_ylabel("Business Driver")

# Add the correlation value to each bar.
for i, v in enumerate(revenue_correlations):
    ax.text(v, i, f"{v:.3f}", va="center")

plt.tight_layout()
plt.show()

revenue_correlations

# 8. Revenue Prediction — Linear Regression

## Business question

> **Can NovaCore predict monthly revenue using operational and commercial drivers?**

> **Important distinction**:
A driver-based regression can predict revenue when the relevant business drivers are available. However, using regression to forecast a future period requires the future values of those drivers to be known or forecast separately.

Regression uses explanatory variables to estimate a target.

> In this lab, the 2025 driver values are available in the test dataset. Therefore, the regression results demonstrate revenue prediction conditional on the observed business drivers. They should not be interpreted as a directly comparable 2025 forecast.


- **Target (`y`)** = revenue.
- **Predictors (`X`)** = selected business drivers.

In [ ]:
# Define the target variable to be predicted.
target = "revenue_CUR"

# Define the business variables used to predict the target.
predictors = [
    "customer_orders",
    "active_customers",
    "customer_retention_pct",
    "sales_pipeline_CUR",
    "marketing_spend_CUR",
    "capacity_utilisation_pct",
    "month_index"
]

print("Target:", target)
print("Predictors:", predictors)

## 8.1 Time-aware train/test split

Use:

- **2021–2024** for training;
- **2025** as the unseen test period.

The model learns from the past and is evaluated on a future period.

In [ ]:
# Split the data chronologically: 2021–2024 for training and 2025 for testing.
train_data = df[df["month"] < "2025-01-01"].copy()
test_data = df[df["month"] >= "2025-01-01"].copy()

print("Training observations:", len(train_data))
print("Test observations:", len(test_data))

In [ ]:
# Separate predictors (X) from the target (y) for training and testing.
X_train = train_data[predictors]
y_train = train_data[target]
X_test = test_data[predictors]
y_test = test_data[target]

In [ ]:
# Create the linear regression model.
revenue_model = LinearRegression()

# Train the model using the training predictors (X) and target (y).
revenue_model.fit(X_train, y_train)

print("Regression model trained successfully.")

In [ ]:
# Create a table showing each predictor and its estimated coefficient.
coefficient_table = pd.DataFrame({
    "Predictor": predictors,
    "Coefficient": revenue_model.coef_
})

coefficient_table

> **Note: Interpreting regression coefficients**
A regression coefficient shows the relationship between a predictor and revenue **after accounting for the other predictors in the model**. Therefore, its sign may differ from the simple correlation between that predictor and revenue.

> For example, active_customers has a positive simple correlation with revenue (0.335), but its regression coefficient is negative. This can occur when predictors are related to each other. The coefficient should therefore **not be interpreted in isolation** or taken to mean that increasing active customers reduces revenue.

In [ ]:
# Generate predictions for training observations and unseen 2025 observations.
train_predictions = revenue_model.predict(X_train)
test_predictions = revenue_model.predict(X_test)

## 8.2 Regression — full-period view

This chart provides the context view: the complete 2021–2025 revenue history, with regression predictions shown for the unseen 2025 period.

In [ ]:
# Compare actual revenue with regression predictions across the full period.
plt.figure(figsize=(13, 5.5))
plt.plot(df["month"], df["revenue_CUR"], label="Actual revenue", marker="o")
plt.plot(test_data["month"], test_predictions, label="Regression prediction", marker="o")

# Mark the beginning of the unseen 2025 test period.
plt.axvline(pd.Timestamp("2025-01-01"), linestyle="--", linewidth=1)

plt.title("Revenue Prediction — Full Historical Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
)

plt.legend()

# Use the same readable month format as the earlier charts.
ticks = df["month"].iloc[::2]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()

## 8.3 Regression — 2025 zoom


> **Evaluating the regression prediction**
The model is evaluated against the unseen 2025 revenue observations using the observed 2025 business-driver values. These results show how accurately the regression predicts revenue when the drivers are available.

> **Important**: this is not equivalent to a genuine future revenue forecast, because future business-driver values would normally need to be known or forecast separately.

In [ ]:
# Zoom in on the unseen 2025 test period.
plt.figure(figsize=(12, 5))
plt.plot(test_data["month"], y_test, marker="o", label="Actual revenue")
plt.plot(test_data["month"], test_predictions, marker="o", label="Regression prediction")

plt.title("Revenue Prediction — 2025 Test Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
)

plt.legend()

# Show every month using the same "Mon YYYY" format.
ticks = test_data["month"].iloc[::1]
plt.xticks(ticks, ticks.dt.strftime("%b %Y"), rotation=45, ha="right")

plt.tight_layout()


# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()

In [ ]:
# Evaluate the regression model using the unseen 2025 test period.
r2 = r2_score(y_test, test_predictions)                    # Explained variation
mae = mean_absolute_error(y_test, test_predictions)        # Average absolute error
rmse = mean_squared_error(y_test, test_predictions) ** 0.5 # Penalises larger errors
mape = np.mean(np.abs((y_test - test_predictions) / y_test)) * 100  # Percentage error

# Present the metrics in a compact results table.
regression_results = pd.DataFrame({
    "Metric": ["R²", "MAE (CUR)", "RMSE (CUR)", "MAPE (%)"],
    "Result": [r2, mae, rmse, mape]
})

regression_results

# 9. Revenue Forecasting

Regression and forecasting answer different questions.


> **Regression**: Which business factors are associated with revenue, and how accurately can revenue be predicted when those drivers are available?

> **Forecasting**: What is likely to happen next based on the historical revenue pattern?

>> The regression model uses business drivers and is evaluated separately.

>> The forecasting models generate future revenue estimates from the historical revenue series.

For forecasting, we will compare:

Seasonal Naïve
Holt-Winters
Prophet

All three forecasting models use the same unseen 2025 test period.


# 10. Seasonal Naïve — Baseline

For monthly data with annual seasonality:

> **January 2025 forecast = January 2024 revenue**

A more sophisticated model should demonstrate that it adds value beyond this simple benchmark.

In [ ]:
# Use 2021–2024 as the forecasting training period.
revenue_train = df[df["month"] < "2025-01-01"]["revenue_CUR"]

# Use January–December 2024 as forecasts for January–December 2025.
seasonal_naive_predictions = revenue_train.iloc[-12:].values

In [ ]:
# Compare the Seasonal Naïve forecast with actual revenue.
plt.figure(figsize=(13, 5.5))
plt.plot(df["month"], df["revenue_CUR"], label="Actual revenue", marker="o")
plt.plot(test_data["month"], seasonal_naive_predictions, label="Seasonal Naïve", marker="o")

# Mark the beginning of the unseen 2025 test period.
plt.axvline(pd.Timestamp("2025-01-01"), linestyle="--", linewidth=1)

plt.title("Seasonal Naïve Forecast — Full Historical Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
)

plt.legend()

ticks = df["month"].iloc[::2]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()

In [ ]:
# Zoom in on the Seasonal Naïve forecast for the unseen 2025 period.
plt.figure(figsize=(12, 5))
plt.plot(test_data["month"], y_test, marker="o", label="Actual revenue")
plt.plot(test_data["month"], seasonal_naive_predictions, marker="o", label="Seasonal Naïve")

plt.title("Seasonal Naïve Forecast — 2025 Test Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
)

plt.legend()

ticks = test_data["month"].iloc[::1]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )

plt.show()

In [ ]:
# Evaluate the Seasonal Naïve baseline on the unseen 2025 period.
baseline_mae = mean_absolute_error(y_test, seasonal_naive_predictions)
baseline_rmse = mean_squared_error(y_test, seasonal_naive_predictions) ** 0.5
baseline_mape = np.mean(np.abs((y_test - seasonal_naive_predictions) / y_test)) * 100

# Store the baseline results for comparison with other forecasting models.
baseline_results = pd.DataFrame({
    "Metric": ["MAE (CUR)", "RMSE (CUR)", "MAPE (%)"],
    "Result": [baseline_mae, baseline_rmse, baseline_mape]
})

baseline_results

# 11. Holt-Winters Forecasting

Holt-Winters is appropriate when trend and seasonality are important.

For NovaCore:

- observations are monthly;
- the seasonal cycle is 12 months;
- an additive trend and additive seasonal component are used.

In [ ]:
# Build Holt-Winters with additive trend and seasonality for monthly data.
holt_winters_model = ExponentialSmoothing(
    revenue_train,
    trend="add",
    seasonal="add",
    seasonal_periods=12
).fit()

# Forecast the 12 unseen months of 2025.
holt_winters_predictions = holt_winters_model.forecast(12)

holt_winters_predictions

In [ ]:
# Compare Holt-Winters forecasts with actual revenue across the full period.
plt.figure(figsize=(13, 5.5))
plt.plot(df["month"], df["revenue_CUR"], label="Actual revenue", marker="o")
plt.plot(test_data["month"], holt_winters_predictions, label="Holt-Winters", marker="o")

# Mark the beginning of the unseen 2025 forecast period.
plt.axvline(pd.Timestamp("2025-01-01"), linestyle="--", linewidth=1)

plt.title("Holt-Winters Forecast — Full Historical Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
)

plt.legend()

ticks = df["month"].iloc[::2]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()

In [ ]:
# Zoom in on the Holt-Winters forecast for the unseen 2025 period.
plt.figure(figsize=(12, 5))
plt.plot(test_data["month"], y_test, marker="o", label="Actual revenue")
plt.plot(test_data["month"], holt_winters_predictions, marker="o", label="Holt-Winters")

plt.title("Holt-Winters Forecast — 2025 Test Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
)

plt.legend()

ticks = test_data["month"].iloc[::1]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()

In [ ]:
# Evaluate Holt-Winters on the same unseen 2025 period.
hw_mae = mean_absolute_error(y_test, holt_winters_predictions)
hw_rmse = mean_squared_error(y_test, holt_winters_predictions) ** 0.5
hw_mape = np.mean(np.abs((y_test - holt_winters_predictions) / y_test)) * 100

# Store the results for comparison with other forecasting models.
hw_results = pd.DataFrame({
    "Metric": ["MAE (CUR)", "RMSE (CUR)", "MAPE (%)"],
    "Result": [hw_mae, hw_rmse, hw_mape]
})

hw_results

# 12. Prophet Forecasting

Prophet provides an alternative time-series forecasting approach.

It is included to answer:

> **Does an alternative model provide useful additional forecasting performance?**

Prophet is not assumed to be the winner. It will be judged using the same unseen 2025 test period and the same evaluation metrics.

**important note:** if you skipped the Prophet installation in Section 3: Do not run any code cells in Section 12. Continue at Section 13: Compare the Forecasting Models. You will compare Seasonal Naïve and Holt–Winters using the same 2025 test period. You can review the Prophet example in the read-only walkthrough.

## 12.1 Prepare data for Prophet

Prophet expects:

- `ds` = date;
- `y` = target value.

In [ ]:
# Convert the training data to Prophet's required ds (date) and y (value) format.
prophet_train = train_data[["month", "revenue_CUR"]].rename(
    columns={"month": "ds", "revenue_CUR": "y"}
)

prophet_train.head()

In [ ]:
# Create Prophet with annual seasonality identified from the decomposition.
prophet_model = Prophet(
    yearly_seasonality=True,   # Capture recurring yearly patterns.
    weekly_seasonality=False,  # Disable weekly patterns for monthly data.
    daily_seasonality=False    # Disable daily patterns for monthly data.
)

# Train the model using the prepared historical data.
prophet_model.fit(prophet_train)

print("Prophet model trained successfully.")

In [ ]:
# Create 12 monthly dates for the 2025 forecast period.
future_dates = prophet_model.make_future_dataframe(
    periods=12,
    freq="ME",
    include_history=True
)

# Generate predictions for the historical and future dates.
prophet_forecast = prophet_model.predict(future_dates)

# Keep only the unseen 2025 period for model evaluation.
prophet_test = prophet_forecast[
    prophet_forecast["ds"] >= pd.Timestamp("2025-01-01")
].copy()

# Extract Prophet's point forecasts for evaluation.
prophet_predictions = prophet_test["yhat"].values

prophet_test[["ds", "yhat", "yhat_lower", "yhat_upper"]]

## 12.2 Prophet: full-period view

The full-period chart provides context and includes Prophet's uncertainty interval.

In [ ]:
# Compare Prophet's forecast and uncertainty with actual revenue.
plt.figure(figsize=(13, 5.5))
plt.plot(df["month"], df["revenue_CUR"], label="Actual revenue", marker="o")
plt.plot(prophet_forecast["ds"], prophet_forecast["yhat"], label="Prophet", marker="o")

# Show Prophet's uncertainty interval around the point forecast.
plt.fill_between(
    prophet_forecast["ds"],
    prophet_forecast["yhat_lower"],
    prophet_forecast["yhat_upper"],
    alpha=0.3,
    label="Prophet uncertainty interval"
)

# Mark the beginning of the unseen 2025 forecast period.
plt.axvline(pd.Timestamp("2025-01-01"), linestyle="--", linewidth=1)

plt.title("Prophet Revenue Forecast — Full Historical Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
)

plt.legend()

ticks = df["month"].iloc[::2]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()


## 12.3 Prophet — 2025 zoom

The zoom chart focuses only on the unseen prediction period.

In [ ]:
# Zoom in on the Prophet forecast for the unseen 2025 period.
plt.figure(figsize=(12, 5))
plt.plot(test_data["month"], y_test, marker="o", label="Actual revenue")
plt.plot(test_data["month"], prophet_predictions, marker="o", label="Prophet")

# Show Prophet's uncertainty interval for the test period.
plt.fill_between(
    test_data["month"],
    prophet_test["yhat_lower"],
    prophet_test["yhat_upper"],
    alpha=0.3,
    label="Prophet uncertainty interval"
)

plt.title("Prophet Revenue Forecast — 2025 Test Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")


plt.legend()

ticks = test_data["month"].iloc[::1]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()

> **Note: Comparing forecast uncertainty**
Prophet provides an uncertainty interval for its forecasts. The current implementation does not provide equivalent prediction intervals for the other models. Therefore, uncertainty should be considered when interpreting the Prophet forecast, but the models should not be compared directly on uncertainty from the current outputs alone.

In [ ]:
# Evaluate Prophet's point forecast on the same unseen 2025 period.
prophet_mae = mean_absolute_error(y_test, prophet_predictions)
prophet_rmse = mean_squared_error(y_test, prophet_predictions) ** 0.5
prophet_mape = np.mean(np.abs((y_test - prophet_predictions) / y_test)) * 100

# Store the results for comparison with other forecasting models.
prophet_results = pd.DataFrame({
    "Metric": ["MAE (CUR)", "RMSE (CUR)", "MAPE (%)"],
    "Result": [prophet_mae, prophet_rmse, prophet_mape]
})

prophet_results

# 13. Compare the Forecasting Models

In [ ]:
# Compare the forecasting models completed in this session.
forecast_comparison = pd.DataFrame({
    "Model": ["Seasonal Naïve", "Holt-Winters"],
    "MAE (CUR)": [baseline_mae, hw_mae],
    "RMSE (CUR)": [baseline_rmse, hw_rmse],
    "MAPE (%)": [baseline_mape, hw_mape]
})

if "prophet_results" in globals():
    forecast_comparison.loc[len(forecast_comparison)] = [
        "Prophet", prophet_mae, prophet_rmse, prophet_mape
    ]

forecast_comparison

## 13.1 Full-period comparison

This is the context view showing all three forecasts against the complete historical series.

In [ ]:
# Compare the forecasts you completed across the full historical period.
plt.figure(figsize=(14, 6))
plt.plot(df["month"], df["revenue_CUR"], marker="o", label="Actual revenue")
plt.plot(test_data["month"], seasonal_naive_predictions, marker="o", label="Seasonal Naïve")
plt.plot(test_data["month"], holt_winters_predictions, marker="o", label="Holt-Winters")
if "prophet_results" in globals():
    plt.plot(test_data["month"], prophet_predictions, marker="o", label="Prophet")
# Mark the beginning of the unseen 2025 test period.
plt.axvline(pd.Timestamp("2025-01-01"), linestyle="--", linewidth=1)

plt.title("Revenue Forecast Comparison — Full Historical Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

plt.legend()

ticks = df["month"].iloc[::2]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M"))

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )
plt.show()



## 13.2 2025 zoom comparison

This chart focuses only on the prediction period so that differences between the models are easier to inspect.

In [ ]:
from matplotlib.ticker import FuncFormatter

# Compare all three forecasts against actual revenue for the unseen 2025 period.
plt.figure(figsize=(13, 5.5))
plt.plot(test_data["month"], y_test, marker="o", label="Actual revenue")
plt.plot(test_data["month"], seasonal_naive_predictions, marker="o", label="Seasonal Naïve")
plt.plot(test_data["month"], holt_winters_predictions, marker="o", label="Holt-Winters")
if "prophet_results" in globals():
    plt.plot(test_data["month"], prophet_predictions, marker="o", label="Prophet")
plt.title("Revenue Forecast Comparison — 2025 Test Period")
plt.xlabel("Month")
plt.ylabel("Revenue (CUR)")

plt.legend()

ticks = test_data["month"].iloc[::1]
plt.xticks(ticks, ticks.dt.strftime("%b %y"), rotation=45, ha="right")

plt.tight_layout()

# Format revenue in millions with two decimal places.
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e6:,.2f}M")
)

# Add minor gridlines.
plt.minorticks_on()
plt.grid(which="minor", linestyle="-", color="#E6A0A0", alpha=0.65)

# Add your organization name to the lower right part. As an example "ACCA"
plt.text(0.99, 0.01, "ACCA", transform=plt.gca().transAxes, ha="right", va="bottom",fontsize=12, fontweight="bold", fontstyle="italic", color="#E31837" )

plt.show()


### Interpretation

Do not select a model simply because it is more sophisticated.

Consider:

- accuracy;
- consistency;
- interpretability;
- uncertainty;
- forecast purpose;
- ease of communication to management.

# 14. Validation and Model Comparison

> **Note: Interpreting model performance across metrics**
Different evaluation metrics may identify different models as stronger. For example, one model may have the lower **MAE**, while another has the lower **RMSE**. This is because the metrics measure error in different ways. Model selection should therefore consider the **purpose of the forecast and the pattern of errors**, rather than relying on a single metric.

> The 2025 period was kept outside model training and is used as the common evaluation period.
However, the models do not use the same information to produce their predictions.

> **Linear Regression** uses the observed 2025 values of the business drivers. Its results therefore show how accurately revenue can be predicted when those drivers are available.

> **Seasonal Naïve, Holt-Winters and Prophet** generate forecasts from the historical revenue series without using the observed 2025 business-driver values.

Therefore, the regression results are included for comparison and learning purposes, but **should not be interpreted as directly comparable with the three time-series forecasts**.

A genuine future revenue forecast using regression would require the future business-driver values to be known or forecast separately.

For time-series problems, a stronger validation strategy can use expanding windows:

```text
Fold 1: 2021 → 2022
Fold 2: 2021 → 2023
Fold 3: 2021 → 2024
Final test: 2025
```

The principle is:

> Earlier observations are used to predict later observations; future observations should not be used to predict the past.

In [ ]:
# Combine the results available in this session.
validation_summary = pd.DataFrame({
    "Approach": ["Linear Regression", "Seasonal Naïve", "Holt-Winters"],
    "Test period": ["2025"] * 3,
    "MAE (CUR)": [mae, baseline_mae, hw_mae],
    "RMSE (CUR)": [rmse, baseline_rmse, hw_rmse],
    "MAPE (%)": [mape, baseline_mape, hw_mape]
})

if "prophet_results" in globals():
    validation_summary.loc[len(validation_summary)] = [
        "Prophet", "2025", prophet_mae, prophet_rmse, prophet_mape
    ]

# Add the information basis used by each approach.
validation_summary["Comparison basis"] = [
    "Conditional prediction using observed 2025 drivers",
    "Time-series forecast",
    "Time-series forecast",
    "Time-series forecast"
]

validation_summary

## **Comparability note**

The regression result cannot be compared directly with the three forecasting models as a future-forecast accuracy measure. Regression uses the actual 2025 values of its business drivers, whereas the time-series models forecast 2025 without access to those future driver values.

The regression result is therefore useful for understanding **conditional prediction performance**, while the three time-series models provide the appropriate comparison for **revenue forecasting from historical revenue patterns**.

# 15. Finance Decision

NovaCore's Finance Director is preparing the next planning cycle.

The Director needs a revenue forecast and wants to understand the business factors associated with revenue.

### Recommendation task

Address:

1. Which forecasting model would you recommend?
2. What evidence supports your choice?
3. How large is the typical forecast error?
4. Which business drivers appear associated with revenue?
5. What limitations should management consider?
6. Would you use the same model for every future forecasting problem?

Your recommendation should combine:

**model evidence + financial interpretation + professional judgement**

The most sophisticated model is not automatically the best model.

> For the forecasting decision, compare the three time-series forecasting models. Use the regression results separately to assess the value of business-driver information

# 16. Reflection Questions

1. Why is decomposition useful before building a forecasting model?
2. Why is randomly shuffling monthly financial data potentially misleading?
3. What does a positive regression coefficient mean in NovaCore's revenue model?
4. Why should a forecasting model be compared with a simple baseline?
5. If a more complex model has the lowest MAPE, should NovaCore automatically select it? Why or why not?
6. Why is an uncertainty interval useful when communicating a forecast?
7. Why might a model that performs well on historical data perform less well in the future?

# 17. Summary and Key Takeaways

You followed the complete predictive modelling workflow:

**Explore → Decompose → Predict / Forecast → Validate → Compare → Decide**

### Decomposition
Revenue was separated into trend, seasonality and residual variation.

### Regression
Business drivers were used to predict revenue and evaluated using R², MAE, RMSE and MAPE.

### Forecasting
The forecasting approaches available in your environment were compared:
- Seasonal Naïve;
- Holt-Winters;
- Prophet.

### Validation
The approaches were evaluated on an unseen future period.

### Decision
Model evidence was translated into a finance recommendation.
The model supports professional judgement; it does not replace it.

# 18. Optional Extensions
### **Extension 1: From Revenue Forecasting to Cash-Flow Insight**
Explore whether expected revenue growth is likely to translate into stronger cash generation. Use revenue, operating costs, and available working-capital information to create a simplified operating cash-flow view. Investigate situations where revenue increases but cash flow does not, and explain the implications for financial planning.

### **Extension 2: Prophet Tuning**
Experiment with selected Prophet settings and assess whether the changes improve out-of-sample forecasting performance. Focus on whether additional model complexity produces a meaningful improvement rather than simply achieving a better fit to historical data.

### **Extension 3 — ARIMA/SARIMA**
For an advanced challenge, compare ARIMA/SARIMA with Holt-Winters and Prophet. Evaluate the models using the same test period and forecasting metrics, and explain which approach provides the most useful evidence for the business.

**Extension principle:**

More complexity should only be preferred when it provides useful additional value.

A technically more sophisticated model is **not automatically a better model**. Students should consider **accuracy, interpretability, complexity, and business usefulness** when deciding whether an extension adds value.

This gives the extensions a stronger progression:
**Finance insight → Better predictors → Model improvement → Advanced forecasting**
